# San Fernando ITT - Geovisor Interactivo

**Flujo de trabajo:**
1. Configuracion del entorno (Colab)
2. Carga y transformacion a WGS84
3. Validacion y exportacion a GeoJSON
4. Geovisor interactivo (poligono editable)
5. Definir area de estudio
6. (OPCIONAL) Subir poligono editado
7. Extraccion de POIs de OpenStreetMap
8. Tabla de POIs
9. Mapa final con POIs
10. Exportacion de resultados

## 1. Configuracion del entorno

In [12]:
import sys
import os
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    os.system(f'{sys.executable} -m pip install -q mapclassify 2>/dev/null')
    os.system('rm -rf Corredor_San_fernando')
    os.system('git clone --depth 1 -q https://github.com/j0rg3c45/Corredor_San_fernando.git')
    ruta_base = Path('Corredor_San_fernando')
else:
    ruta_base = Path('..')

print(f"Entorno: {'Google Colab' if EN_COLAB else 'Local'}")
print(f"Ruta base: {ruta_base.resolve()}")

Entorno: Google Colab
Ruta base: /content/Corredor_San_fernando


## 2. Importaciones y rutas

In [13]:
import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import Draw
import json
import requests

# Rutas del proyecto
ruta_geojson_fuente = ruta_base / 'Data' / 'Geojson_Corredor_San_fernando' / 'geojson_Corredor_San_fernando.geojson'
ruta_shapefile = ruta_base / 'Data' / 'shape_geo' / 'Corredor_San_fernando.shp'
ruta_geojson_wgs84 = ruta_base / 'Data' / 'Corredor_San_fernando_wgs84.geojson'
ruta_outputs = ruta_base / 'Outputs'
ruta_outputs.mkdir(parents=True, exist_ok=True)

# Priorizar GeoJSON actualizado sobre shapefile
if ruta_geojson_fuente.exists():
    ruta_entrada = ruta_geojson_fuente
elif ruta_shapefile.exists():
    ruta_entrada = ruta_shapefile
else:
    raise FileNotFoundError('No se encontro archivo de entrada en Data/')

print(f"Archivo de entrada: {ruta_entrada}")

Archivo de entrada: Corredor_San_fernando/Data/Geojson_Corredor_San_fernando/geojson_Corredor_San_fernando.geojson


## 3. Carga, transformacion y validacion

In [14]:
# Cargar datos del corredor
gdf_corredor = gpd.read_file(ruta_entrada)
print(f"CRS original corredor: {gdf_corredor.crs}")

# Asignar CRS si no tiene
if gdf_corredor.crs is None:
    gdf_corredor = gdf_corredor.set_crs(epsg=3115)

# Reproyectar a WGS84
if gdf_corredor.crs.to_epsg() != 4326:
    gdf_corredor = gdf_corredor.to_crs(epsg=4326)

# Validar geometrias
invalidas = gdf_corredor[~gdf_corredor.geometry.is_valid]
if len(invalidas) > 0:
    gdf_corredor['geometry'] = gdf_corredor.geometry.buffer(0)
    print(f"Corregidas {len(invalidas)} geometrias invalidas.")

# Exportar GeoJSON en WGS84
gdf_corredor.to_file(ruta_geojson_wgs84, driver='GeoJSON')

# Cargar shapefile yawa
ruta_yawa = ruta_base / 'Data' / 'shape_geo' / 'yawa.shp'
if ruta_yawa.exists():
    gdf_yawa = gpd.read_file(ruta_yawa)
    if gdf_yawa.crs is None:
        gdf_yawa = gdf_yawa.set_crs(epsg=3115)
    if gdf_yawa.crs.to_epsg() != 4326:
        gdf_yawa = gdf_yawa.to_crs(epsg=4326)
    # Exportar yawa como GeoJSON
    ruta_yawa_geojson = ruta_base / 'Data' / 'yawa_wgs84.geojson'
    gdf_yawa.to_file(ruta_yawa_geojson, driver='GeoJSON')
    print(f"Yawa cargado: {len(gdf_yawa)} registros, CRS: {gdf_yawa.crs}")
else:
    gdf_yawa = None
    print("Shapefile yawa no encontrado.")

print(f"\nCorredor - CRS final: {gdf_corredor.crs}")
print(f"Corredor - Bounds: {gdf_corredor.total_bounds}")
gdf_corredor.head()

CRS original corredor: EPSG:3115
Yawa cargado: 1 registros, CRS: EPSG:4326

Corredor - CRS final: EPSG:4326
Corredor - Bounds: [-76.54983987   3.41250651 -76.53276244   3.43406679]


,Id,geometry
0,0,"MULTIPOLYGON (((-76.53468 3.42265, -76.53535 3..."


## 4. Geovisor interactivo (poligono editable)

El poligono se carga dentro del FeatureGroup editable.
- Haz clic en el icono de edicion (lapiz) para mover vertices
- Haz clic en Save cuando termines
- Haz clic en Export para descargar el poligono editado
- Sube ese archivo en la celda 6 (opcional)

In [15]:
# Leer GeoJSON
with open(ruta_geojson_wgs84) as f:
    geojson_data = json.load(f)

# Centro del mapa
centroide = gdf_corredor.union_all().centroid
centro = [centroide.y, centroide.x]

# Crear mapa
mapa = folium.Map(location=centro, zoom_start=15, tiles='OpenStreetMap')
folium.TileLayer('CartoDB positron', name='CartoDB Claro').add_to(mapa)
folium.TileLayer('CartoDB dark_matter', name='CartoDB Oscuro').add_to(mapa)

# Capa de yawa (si existe)
if gdf_yawa is not None:
    folium.GeoJson(
        gdf_yawa.to_json(),
        name='Yawa',
        style_function=lambda x: {
            'fillColor': '#ff7800',
            'color': '#ff7800',
            'weight': 2,
            'fillOpacity': 0.25
        }
    ).add_to(mapa)

# FeatureGroup editable (corredor)
fg_editable = folium.FeatureGroup(name='Poligono editable')
fg_editable.add_to(mapa)

# Draw plugin
Draw(
    export=True,
    draw_options={'polyline': False, 'rectangle': True, 'polygon': True,
                  'circle': False, 'marker': False, 'circlemarker': False},
    edit_options={'edit': True, 'remove': True}
).add_to(mapa)

# Inyectar JS para cargar poligono en el FeatureGroup editable
geojson_str = json.dumps(geojson_data)
js_code = f"""
<script>
setTimeout(function() {{
    var map = null;
    document.querySelectorAll('.folium-map').forEach(function(el) {{
        if (window[el.id]) map = window[el.id];
    }});
    if (!map) return;
    var drawnItems = null;
    for (var key in window) {{
        if (key.startsWith('drawnItems_')) {{ drawnItems = window[key]; break; }}
    }}
    if (drawnItems) {{
        L.geoJSON({geojson_str}).eachLayer(function(layer) {{
            drawnItems.addLayer(layer);
        }});
    }}
}}, 1000);
</script>
"""
mapa.get_root().html.add_child(folium.Element(js_code))
folium.LayerControl(collapsed=False).add_to(mapa)

# Guardar HTML
ruta_mapa = ruta_outputs / 'geovisor_editable.html'
mapa.save(str(ruta_mapa))
print(f"Mapa guardado: {ruta_mapa}")
mapa

Mapa guardado: Corredor_San_fernando/Outputs/geovisor_editable.html


## 5. Definir area de estudio

Se usa el poligono original. Si quieres usar un poligono editado,
ejecuta la celda 6 (opcional) ANTES de esta.

In [16]:
# Usar poligono original como area de estudio
# Si ejecutaste la celda 6 y subiste un poligono, esta variable ya estara definida
if 'gdf_area_estudio' not in dir():
    gdf_area_estudio = gdf_corredor.copy()

print(f"Area de estudio: {len(gdf_area_estudio)} poligono(s)")
print(f"Bounds: {gdf_area_estudio.total_bounds}")

Area de estudio: 1 poligono(s)
Bounds: [-76.54983987   3.41250651 -76.53276244   3.43406679]


## 6. (OPCIONAL) Subir poligono editado

Ejecuta esta celda SOLO si editaste el poligono en el mapa y quieres usarlo.
Si no, **saltala** y ve directo a la seccion 7.

In [17]:
# Cambiar a True SOLO si quieres subir un poligono editado
SUBIR_POLIGONO = False

if SUBIR_POLIGONO and EN_COLAB:
    from google.colab import files
    print("Sube el GeoJSON editado (data.geojson):")
    uploaded = files.upload()
    if uploaded:
        nombre_archivo = list(uploaded.keys())[0]
        contenido = uploaded[nombre_archivo].decode('utf-8')
        geojson_editado = json.loads(contenido)
        gdf_area_estudio = gpd.GeoDataFrame.from_features(
            geojson_editado['features'], crs='EPSG:4326'
        )
        print(f"Poligono editado cargado: {len(gdf_area_estudio)} geometria(s)")
        print(f"Bounds: {gdf_area_estudio.total_bounds}")
else:
    print("Usando poligono original (SUBIR_POLIGONO = False).")

Usando poligono original (SUBIR_POLIGONO = False).


## 7. Extraccion de POIs de OpenStreetMap

Se usa osmnx para extraer POIs directamente dentro del poligono del area de estudio.

In [18]:
import os
os.system(f'{sys.executable} -m pip install -q osmnx')
import osmnx as ox

# Tags de OSM a consultar por categoria
tags_por_categoria = {
    'Salud': {'amenity': ['hospital', 'clinic', 'pharmacy', 'doctors']},
    'Educacion': {'amenity': ['school', 'university', 'kindergarten', 'college']},
    'Comercio': {'shop': True},
    'Transporte': {'amenity': ['bus_station', 'fuel', 'parking'],
                   'highway': 'bus_stop'},
    'Recreacion': {'leisure': ['park', 'sports_centre', 'playground']},
    'Servicios': {'amenity': ['bank', 'police', 'fire_station', 'post_office']}
}

# Obtener el poligono del area de estudio
poligono = gdf_area_estudio.union_all()

# Extraer POIs por categoria
lista_pois = []
for categoria, tags in tags_por_categoria.items():
    try:
        gdf_cat = ox.features_from_polygon(poligono, tags=tags)
        if len(gdf_cat) > 0:
            # Obtener centroide para geometrias no puntuales
            gdf_cat['geometry'] = gdf_cat.geometry.centroid
            gdf_cat['categoria'] = categoria
            gdf_cat['nombre'] = gdf_cat.get('name', 'Sin nombre')
            gdf_cat['nombre'] = gdf_cat['nombre'].fillna('Sin nombre')
            gdf_cat['latitud'] = gdf_cat.geometry.y
            gdf_cat['longitud'] = gdf_cat.geometry.x
            lista_pois.append(gdf_cat[['nombre', 'categoria', 'latitud', 'longitud', 'geometry']])
            print(f"{categoria}: {len(gdf_cat)} POIs")
    except Exception as e:
        print(f"{categoria}: sin resultados ({e})")

# Unir todos los POIs
if lista_pois:
    gdf_pois_dentro = pd.concat(lista_pois, ignore_index=True)
    gdf_pois_dentro = gpd.GeoDataFrame(gdf_pois_dentro, geometry='geometry', crs='EPSG:4326')
    print(f"\nTotal POIs encontrados: {len(gdf_pois_dentro)}")
else:
    gdf_pois_dentro = gpd.GeoDataFrame()
    print("No se encontraron POIs.")

/tmp/ipykernel_2076/2963135785.py:26: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_cat['geometry'] = gdf_cat.geometry.centroid


Salud: 137 POIs


/tmp/ipykernel_2076/2963135785.py:26: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_cat['geometry'] = gdf_cat.geometry.centroid


Educacion: 9 POIs


/tmp/ipykernel_2076/2963135785.py:26: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_cat['geometry'] = gdf_cat.geometry.centroid


...

/tmp/ipykernel_2076/2963135785.py:26: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_cat['geometry'] = gdf_cat.geometry.centroid


Transporte: 58 POIs


/tmp/ipykernel_2076/2963135785.py:26: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_cat['geometry'] = gdf_cat.geometry.centroid


Recreacion: 16 POIs
Servicios: 14 POIs

Total POIs encontrados: 273


/tmp/ipykernel_2076/2963135785.py:26: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_cat['geometry'] = gdf_cat.geometry.centroid


## 8. Filtrar POIs dentro del poligono y mostrar tabla

In [19]:
if len(gdf_pois_dentro) > 0:
    # Tabla
    tabla = gdf_pois_dentro[['nombre', 'categoria', 'latitud', 'longitud']].copy()
    tabla = tabla.sort_values(['categoria', 'nombre']).reset_index(drop=True)
    tabla.index = tabla.index + 1
    tabla.index.name = 'N'
    print(f"Total: {len(tabla)} puntos de interes")
    print()
    print("Por categoria:")
    print(gdf_pois_dentro['categoria'].value_counts().to_string())
    print()
    display(tabla)
else:
    print("No hay POIs para mostrar.")

Total: 273 puntos de interes

Por categoria:
categoria
Salud         137
Transporte     58
Comercio       39
Recreacion     16
Servicios      14
Educacion       9



,nombre,categoria,latitud,longitud
N,,,,
1,Almacenes Éxito,Comercio,3.425428,-76.545384
2,Amodin,Comercio,3.420031,-76.537818
3,Buen Viaje Agencia de Viajes y Turismo,Comercio,3.432763,-76.542823
4,Calicentro,Comercio,3.420436,-76.542400
5,Centro Empresarial,Comercio,3.421669,-76.541627
...,...,...,...,...
269,Sin nombre,Transporte,3.419225,-76.539693
270,Sin nombre,Transporte,3.422147,-76.540731
271,Sin nombre,Transporte,3.422099,-76.540832


## 9. Mapa final con POIs clasificados

In [22]:
colores = {'Salud': 'red', 'Educacion': 'blue', 'Comercio': 'green',
           'Transporte': 'orange', 'Recreacion': 'purple', 'Servicios': 'darkblue'}

mapa_final = folium.Map(location=centro, zoom_start=15, tiles='CartoDB positron')

# Poligono del area
folium.GeoJson(
    gdf_area_estudio.to_json(),
    name='Area de estudio',
    style_function=lambda x: {'fillColor': '#3388ff', 'color': '#2255cc',
                               'weight': 2, 'fillOpacity': 0.1}
).add_to(mapa_final)

# Capa yawa
if gdf_yawa is not None:
    folium.GeoJson(
        gdf_yawa.to_json(),
        name='Yawa',
        style_function=lambda x: {'fillColor': '#ff7800', 'color': '#ff7800',
                                   'weight': 2, 'fillOpacity': 0.2}
    ).add_to(mapa_final)

# POIs por categoria
if len(gdf_pois_dentro) > 0:
    for cat, color in colores.items():
        pois_cat = gdf_pois_dentro[gdf_pois_dentro['categoria'] == cat]
        if len(pois_cat) == 0:
            continue
        fg = folium.FeatureGroup(name=f"{cat} ({len(pois_cat)})")
        for _, poi in pois_cat.iterrows():
            folium.CircleMarker(
                location=[poi.latitud, poi.longitud],
                radius=6, color=color, fill=True,
                fill_color=color, fill_opacity=0.7,
                popup=f"<b>{poi.nombre}</b><br>{poi.categoria}",
                tooltip=poi.nombre
            ).add_to(fg)
        fg.add_to(mapa_final)

folium.LayerControl(collapsed=False).add_to(mapa_final)
mapa_final

AttributeError: 'Series' object has no attribute 'subtipo'

## 10. Exportacion de resultados

In [ ]:
from google.colab import files as colab_files

if len(gdf_pois_dentro) > 0:
    # GeoJSON de POIs
    ruta_pois_gj = ruta_outputs / 'pois_area_estudio.geojson'
    gdf_pois_dentro.to_file(ruta_pois_gj, driver='GeoJSON')
    print(f"POIs GeoJSON: {ruta_pois_gj}")

    # CSV de POIs
    ruta_pois_csv = ruta_outputs / 'pois_area_estudio.csv'
    gdf_pois_dentro.drop(columns='geometry').to_csv(ruta_pois_csv, index=False)
    print(f"POIs CSV: {ruta_pois_csv}")

# Mapa HTML
ruta_html = ruta_outputs / 'geovisor_pois_san_fernando.html'
mapa_final.save(str(ruta_html))
print(f"Mapa HTML: {ruta_html}")

# Descargar en Colab
if EN_COLAB:
    if len(gdf_pois_dentro) > 0:
        colab_files.download(str(ruta_pois_csv))
    colab_files.download(str(ruta_html))

Mapa HTML: Corredor_San_fernando/Outputs/geovisor_pois_san_fernando.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>